# Indian TTS — Train Custom Indian English Voices (Male & Female)

**What this notebook does:**
1. Downloads Indian English speech data (legally safe — CC-0 & CC-BY 4.0 only)
2. Preprocesses audio for training
3. Trains a VITS2 model from scratch on A100 GPU
4. Generates Indian English speech with male/female voice selection

**Requirements:** Google Colab Pro+ with A100 GPU runtime

**Data Sources (all legally safe for commercial use):**
- Mozilla Common Voice — Indian English subset (CC-0, public domain)
- Google FLEURS — Indian English (CC-BY 4.0)

## Step 0: Verify GPU & Setup Runtime

**IMPORTANT:** Go to `Runtime > Change runtime type` and select **A100 GPU** before running.

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > A100")

## Step 1: Install Dependencies & Clone Repo

In [ ]:
# Clone the repo
!git clone https://github.com/seetha0712/text2speech_1.git /content/indian_tts
%cd /content/indian_tts
!git checkout claude/custom-indian-tts-model-TUAjJ

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

# Install espeak-ng for phonemization
!apt-get install -qq espeak-ng > /dev/null 2>&1

print("\nInstallation complete!")

## Step 2: Download Indian English Data

This downloads **only legally safe** datasets:
- **Common Voice** (CC-0 — public domain, no restrictions)
- **FLEURS** (CC-BY 4.0 — commercial OK with attribution)

You may need to accept Common Voice terms on HuggingFace first:
https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0

In [ ]:
# Login to HuggingFace (needed for Common Voice)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Download Indian English data
# Adjust --max-hours based on how much data you want
# More hours = better quality but longer training

!python -m indian_tts.data.preprocess \
    --source all \
    --output /content/data \
    --max-hours 15 \
    --min-upvotes 2 \
    --target-sr 22050

In [ ]:
# Verify the data
import os

for split in ['train', 'val', 'test']:
    path = f'/content/data/{split}.txt'
    if os.path.exists(path):
        with open(path) as f:
            lines = [l for l in f if l.strip() and not l.startswith('#')]
        print(f"{split}: {len(lines)} samples")
        # Show first 3 entries
        for line in lines[:3]:
            parts = line.strip().split('|')
            gender = 'Male' if parts[1] == '0' else 'Female'
            print(f"  [{gender}] {parts[2][:60]}...")
    else:
        print(f"{split}: NOT FOUND")

## Step 3: Configure Training

The config is optimized for A100 80GB. Adjust `batch_size` if you get OOM errors.

In [ ]:
import yaml

# Load base config
with open('configs/base_config.yaml') as f:
    config = yaml.safe_load(f)

# === Adjust for Colab A100 ===

# Data paths
config['data']['training_files'] = '/content/data/train.txt'
config['data']['validation_files'] = '/content/data/val.txt'

# A100-optimized training params
config['training']['batch_size'] = 48         # A100 80GB can handle this
config['training']['max_steps'] = 200000      # ~2-3 days on A100
config['training']['fp16'] = True             # Use mixed precision
config['training']['save_every_n_steps'] = 5000
config['training']['eval_every_n_steps'] = 2500
config['training']['log_every_n_steps'] = 50

# Output paths (use Colab's /content for speed)
config['paths']['output_dir'] = '/content/outputs'
config['paths']['checkpoint_dir'] = '/content/outputs/checkpoints'
config['paths']['log_dir'] = '/content/outputs/logs'

# Save updated config
colab_config_path = 'configs/colab_a100_config.yaml'
with open(colab_config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config saved to:", colab_config_path)
print(f"\nKey settings:")
print(f"  Batch size:  {config['training']['batch_size']}")
print(f"  Max steps:   {config['training']['max_steps']}")
print(f"  FP16:        {config['training']['fp16']}")
print(f"  Save every:  {config['training']['save_every_n_steps']} steps")

## Step 4: Train the Model

Training time estimates on A100:
- 50K steps: ~6-8 hours (early results, some artifacts)
- 100K steps: ~12-16 hours (decent quality)
- 200K steps: ~24-32 hours (good quality)

**TIP:** Start TensorBoard in another cell to monitor training.

In [ ]:
# Start TensorBoard (run this in parallel while training)
%load_ext tensorboard
%tensorboard --logdir /content/outputs/logs

In [ ]:
# START TRAINING
!python -m indian_tts.train --config configs/colab_a100_config.yaml

In [ ]:
# RESUME TRAINING (if Colab disconnected)
# Find the latest checkpoint
import glob
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_step_*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f"Resuming from: {latest}")
    !python -m indian_tts.train \
        --config configs/colab_a100_config.yaml \
        --resume {latest}
else:
    print("No checkpoints found. Start fresh training first.")

## Step 5: Generate Speech (Inference)

After training, generate Indian English speech with male or female voice.

In [ ]:
from indian_tts.inference import IndianTTS
import IPython.display as ipd

# Load the trained model
# Use the latest checkpoint or 'final'
import glob
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
checkpoint = ckpts[-1] if ckpts else None

if checkpoint:
    print(f"Loading: {checkpoint}")
    tts = IndianTTS(checkpoint)
else:
    print("ERROR: No checkpoint found. Train the model first (Step 4).")

In [ ]:
# Generate FEMALE Indian English voice
text = "Hello, welcome to our text to speech system. The weather in Bangalore is very pleasant today."

audio = tts.synthesize(
    text=text,
    voice="female",
    speed=1.0,            # 1.0 = normal, 0.8 = slower, 1.2 = faster
    expressiveness=0.667, # 0 = monotone, 1 = very expressive
)

print("Female voice:")
ipd.display(ipd.Audio(audio, rate=tts.sampling_rate))

In [ ]:
# Generate MALE Indian English voice
text = "Good morning everyone. Today we will discuss the quarterly results of our company."

audio = tts.synthesize(
    text=text,
    voice="male",
    speed=1.0,
    expressiveness=0.667,
)

print("Male voice:")
ipd.display(ipd.Audio(audio, rate=tts.sampling_rate))

In [ ]:
# Try different texts and settings
test_sentences = [
    "The train from Delhi to Mumbai will depart from platform number three.",
    "India is celebrating its seventy-fifth year of independence this year.",
    "Please submit your assignment before the deadline on Friday.",
    "The monsoon season brings much needed relief from the summer heat.",
    "Our new software product will be launched next quarter.",
]

for i, text in enumerate(test_sentences):
    for voice in ["male", "female"]:
        audio = tts.synthesize(text, voice=voice)
        print(f"\n[{voice.upper()}] {text[:60]}...")
        ipd.display(ipd.Audio(audio, rate=tts.sampling_rate))

## Step 6: Evaluate Quality

In [ ]:
!python -m indian_tts.evaluate \
    --checkpoint {checkpoint} \
    --test-manifest /content/data/test.txt \
    --output-dir /content/outputs/eval \
    --max-samples 30

## Step 7: Save Model to Google Drive

Save your trained model so you don't lose it when Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy checkpoints to Google Drive
import shutil
drive_path = '/content/drive/MyDrive/indian_tts_checkpoints'
os.makedirs(drive_path, exist_ok=True)

# Copy latest checkpoint
if checkpoint:
    dest = os.path.join(drive_path, os.path.basename(checkpoint))
    shutil.copy2(checkpoint, dest)
    print(f"Saved to: {dest}")

# Copy config
shutil.copy2('configs/colab_a100_config.yaml', drive_path)
print("Config saved to Drive.")

## Step 8 (Optional): Export to ONNX for Production

In [ ]:
# Export for deployment
tts.export_onnx('/content/outputs/indian_tts.onnx')
print("ONNX model exported!")